# Tarea 3

## Cuerpo docente

- Profesores: [Andrés Abeliuk](https://aabeliuk.github.io/), Fabrizzio Pezzolla
- Profesor auxiliar: Rodrigo Molina

## Instrucciones generales

- Grupos de máximo 4 personas.
- Está prohibido compartir las respuestas con otros grupos.
- Indicios de copia serán penalizados con la nota mínima.
- Cualquier duda fuera del horario de clases, al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente; si utilizan material extra, deben citarlo.


## Integrantes

> POR FAVOR AGREGAR TODOS LOS NOMBRES DE LOS INTEGRANTES

# 0. Preparación

Esta tarea necesita una GPU. En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

In [ ]:
!pip install -q transformers==5.22.1 trl==1.12.0 peft==0.21.2
!pip install -q bitsandbytes==0.50.2 datasets==4.8.5

In [ ]:
import gc
import json
import re
import sqlite3
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig)
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

SEMILLA = 0
URL_DATOS = ("https://raw.githubusercontent.com/dccuchile/CC66Q/main/"
             "tareas/data/")
pd.set_option("display.max_colwidth", None)  # textos completos

Una empresa guarda sus ventas en una tabla y quiere un asistente que responda preguntas en castellano. El asistente no responde directamente: **escribe la consulta SQL** que responde la pregunta. Así puede contestar preguntas que nadie ha hecho antes, y su respuesta sigue siendo correcta cuando la tabla se actualiza.

Para entrenarlo, la empresa tiene dos tipos de datos:

- **Demostraciones:** 150 preguntas con la consulta que escribió un analista. Son 53 consultas distintas, cada una con hasta tres redacciones de la pregunta. Cubren 12 tipos de pregunta, los *conocidos*: conteos y sumas con filtros de igualdad, fechas y umbrales.
- **Preguntas solo con resultado:** 600 preguntas de las que se conoce la respuesta correcta, pero no la consulta: alguien las respondió una vez a mano, con una planilla o un reporte. Entre ellas hay 172 de 9 tipos *nuevos*, que no aparecen en las demostraciones: promedios, máximos y mínimos, `GROUP BY`, rankings, conteos por estado y exclusiones como “sin contar las anuladas”.

Las 142 preguntas de prueba incluyen 33 de los tipos nuevos. Ninguna consulta se repite entre los tres conjuntos.

En esta tarea van a:

1. Cargar un modelo pequeño en 4 bits y medir qué se gana y qué se pierde (clase 7).
2. Enseñarle el formato con SFT y QLoRA, usando las demostraciones (clase 7).
3. Seguir entrenándolo con GRPO, usando solo los resultados (clase 8).

Entrenar con el resultado de ejecutar la consulta no es una idea nueva: Seq2SQL ([Zhong, Xiong y Socher, 2017](https://arxiv.org/abs/1709.00103)) ya usaba en WikiSQL una recompensa por ejecución: −2 si la consulta no es válida, −1 si da otro resultado y +1 si acierta.

Los datos son sintéticos: una tabla de 500 ventas, con las regiones y los estados en códigos.

La tabla `ventas`:

In [ ]:
ventas = pd.read_csv(URL_DATOS + "ventas.csv")

conexion = sqlite3.connect(":memory:")
ventas.to_sql("ventas", conexion, index=False)
conexion.execute("PRAGMA query_only = ON")  # nadie puede modificar la tabla

ventas.head()

Cada pregunta trae su tipo (`familia`), el resultado correcto (`resultado`) y si el orden de las filas importa (`ordenado`). Las demostraciones y las de prueba traen además la consulta de referencia (`sql`). Las preguntas solo con resultado (`split` igual a `rl`) no la traen. Los tipos que aparecen en las demostraciones son los conocidos, y el resto, los nuevos.

In [ ]:
preguntas = pd.read_json(URL_DATOS + "preguntas_sql.jsonl", lines=True)
demostraciones = preguntas[preguntas["split"] == "demo"]
solo_resultado = preguntas[preguntas["split"] == "rl"]
prueba = preguntas[preguntas["split"] == "prueba"]
tipos_conocidos = set(demostraciones["familia"])

demostraciones.head()

El modelo recibe el esquema de la tabla, el significado de los códigos y la pregunta.

In [ ]:
ESQUEMA = (
    "Tabla ventas(id, fecha, region, producto, vendedor, estado, cantidad, "
    "monto).\n"
    "fecha es texto AAAA-MM-DD; todas las fechas son de 2026.\n"
    "region: RM = Metropolitana, VS = Valparaíso, BI = Biobío, "
    "AN = Antofagasta.\n"
    "estado: P = pendiente, D = despachada, A = anulada.\n"
    "monto está en pesos.")


def mensajes(pregunta: str) -> list[dict]:
    texto = (f"{ESQUEMA}\n\nEscribe una consulta SQLite que responda la "
             f"pregunta. Responde solo con la consulta.\n\n"
             f"Pregunta: {pregunta}")
    return [{"role": "user", "content": texto}]


print(mensajes(prueba["pregunta"].iloc[0])[0]["content"])

`revisar` corre una consulta en la tabla y la compara con el resultado correcto. Devuelve `"falla"` si la consulta no se ejecuta, `"distinto"` si se ejecuta pero da otro resultado y `"correcto"` si da las mismas filas, con las mismas columnas en el mismo orden. El orden de las filas importa solo cuando la pregunta lo pide, y los números se comparan con dos decimales.

In [ ]:
def revisar(texto: str, esperado: list, ordenado: bool) -> str:
    # quita las marcas ```sql y lo que venga después del primer ;
    consulta = re.sub(r"```(?:sql)?", "", texto).split(";")[0].strip()
    if not consulta:
        return "falla"
    inicio = time.time()
    # corta las consultas que tardan más de un segundo
    conexion.set_progress_handler(lambda: time.time() - inicio > 1, 1000)
    try:
        obtenido = conexion.execute(consulta).fetchall()
    except Exception:
        return "falla"
    finally:
        conexion.set_progress_handler(None, 0)

    def numeros(filas: list) -> list:
        return [[round(float(x), 2) if isinstance(x, (int, float)) else x
                 for x in fila] for fila in filas]

    obtenido, esperado = numeros(obtenido), numeros(esperado)
    if not ordenado:
        obtenido = sorted(obtenido, key=repr)
        esperado = sorted(esperado, key=repr)
    return "correcto" if obtenido == esperado else "distinto"


n_anuladas = [[len(ventas[ventas["estado"] == "A"])]]
print(revisar("SELECT COUNT(*) FROM ventas WHERE estado = 'A'",
              n_anuladas, False))
print(revisar("SELECT COUNT(*) FROM ventas", n_anuladas, False))
print(revisar("SELECT COUNT(*) FROM tabla_que_no_existe", n_anuladas,
              False))

# 1. El modelo en 16 y en 4 bits

Usaremos Qwen2.5-0.5B-Instruct, un modelo de 500 millones de parámetros ya ajustado para seguir instrucciones.

`evaluar` responde las 142 preguntas de prueba con decodificación greedy, hasta 64 tokens por respuesta, y revisa cada consulta con `revisar`. Devuelve la exactitud (en total, en los tipos conocidos y en los nuevos), la fracción de consultas que se ejecutan y una tabla con la respuesta (`respuesta`) y la revisión (`revision`) de cada pregunta. Las guardamos en `resultados` y `detalles`, con el nombre de cada modelo.

In [ ]:
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizador = AutoTokenizer.from_pretrained(MODELO)
tokenizador.padding_side = "left"


@torch.no_grad()
def evaluar(modelo: AutoModelForCausalLM | PeftModel
            ) -> tuple[dict, pd.DataFrame]:
    modelo.eval()
    respuestas = []
    for i in range(0, len(prueba), 50):
        textos = [tokenizador.apply_chat_template(
                      mensajes(p), tokenize=False, add_generation_prompt=True)
                  for p in prueba["pregunta"].iloc[i:i + 50]]
        entrada = tokenizador(textos, return_tensors="pt",
                              padding=True).to(0)
        salida = modelo.generate(**entrada, max_new_tokens=64,
                                 do_sample=False, repetition_penalty=1.0,
                                 pad_token_id=tokenizador.pad_token_id)
        respuestas += tokenizador.batch_decode(
            salida[:, entrada.input_ids.shape[1]:], skip_special_tokens=True)

    revisiones = [revisar(r, e, o) for r, e, o in zip(
        respuestas, prueba["resultado"], prueba["ordenado"])]
    detalle = prueba.assign(respuesta=respuestas, revision=revisiones)
    correcta = detalle["revision"] == "correcto"
    conocida = detalle["familia"].isin(tipos_conocidos)
    def tasa(aciertos: pd.Series) -> float:
        return round(float(aciertos.mean()), 3)

    metricas = {
        "exactitud": tasa(correcta),
        "tipos conocidos": tasa(correcta[conocida]),
        "tipos nuevos": tasa(correcta[~conocida]),
        "se ejecuta": tasa(detalle["revision"] != "falla"),
    }
    return metricas, detalle


def liberar_memoria() -> None:
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
modelo_16 = AutoModelForCausalLM.from_pretrained(
    MODELO, device_map={"": 0}, dtype=torch.float16)

resultados, detalles, pesos = {}, {}, {}
resultados["16 bits"], detalles["16 bits"] = evaluar(modelo_16)
pesos["16 bits"] = round(modelo_16.get_memory_footprint() / 2**30, 2)
del modelo_16
liberar_memoria()
resultados["16 bits"]

Completen `config_4bits`, la configuración para cargar el modelo en 4 bits como en la clase 7: NF4, cuantización doble y cálculos en fp16. **(0,4 pts)**

In [ ]:
# TODO 1: configuración de 4 bits
config_4bits = ...


In [ ]:
def cargar_4bits() -> AutoModelForCausalLM:
    return AutoModelForCausalLM.from_pretrained(
        MODELO, quantization_config=config_4bits, device_map={"": 0},
        dtype=torch.float16)


modelo_4 = cargar_4bits()
resultados["4 bits"], detalles["4 bits"] = evaluar(modelo_4)
pesos["4 bits"] = round(modelo_4.get_memory_footprint() / 2**30, 2)
del modelo_4
liberar_memoria()

tabla_bits = pd.DataFrame(resultados).T
tabla_bits["pesos (GB)"] = pd.Series(pesos)
tabla_bits

---

**📝 Pregunta 1 (1,0 pts):** Comparen las dos filas de la tabla. ¿Qué ganaron y qué perdieron al cargar el modelo en 4 bits? El T4 de Colab tiene 15 GB de memoria: ¿cabría en 16 bits un modelo de 13.000 millones de parámetros? ¿Y en 4 bits? Justifiquen con una estimación del tamaño de los pesos. En QLoRA los pesos en 4 bits quedan congelados: ¿qué se entrena, entonces, y con qué precisión?
> **Respuesta:**

---

# 2. SFT

En la clase 7 vimos QLoRA: el modelo se carga en 4 bits y queda congelado, y solo se entrena un adapter LoRA. Aquí lo usaremos para SFT: el modelo aprende a imitar pares de pregunta y consulta escrita por una persona. Entrenaremos dos veces, con 30 y con 150 demostraciones, durante 3 epochs.

Completen `entrenar_sft`, que ajusta un adapter LoRA nuevo con las primeras `n` demostraciones. El modelo es `prepare_model_for_kbit_training(cargar_4bits())` y el adapter se pasa a `SFTTrainer` como `peft_config`, junto con `processing_class=tokenizador`. Cada ejemplo es un `prompt` (`mensajes(pregunta)`) y una `completion` con la consulta como respuesta del asistente: `[{"role": "assistant", "content": consulta}]`. Usen:

- `LoraConfig` con `r=4`, `lora_alpha=8`, `lora_dropout=0.05`, `target_modules="all-linear"` y `task_type="CAUSAL_LM"`.
- `SFTConfig` con 3 epochs, tasa de aprendizaje 2e-4, lotes de 8, `max_length=512`, `fp16=True`, `gradient_checkpointing=True`, `save_strategy="no"`, `report_to="none"` y `seed=SEMILLA`.

**(0,6 pts)**

In [ ]:
def entrenar_sft(n: int) -> PeftModel:
    torch.manual_seed(SEMILLA)  # LoRA parte de valores al azar

    # TODO 2: SFT con las primeras n demostraciones

    entrenador = ...  # Reemplazar por su SFTTrainer

    # con fp16=True, los parámetros que se entrenan deben estar en fp32
    for parametro in entrenador.model.parameters():
        if parametro.requires_grad:
            parametro.data = parametro.data.float()
    entrenador.train()
    return entrenador.model

Entrenemos con 30 demostraciones y con las 150. Guardamos el adapter de 150, del que partirá GRPO.

In [ ]:
tiempos = {}
for n in [30, 150]:
    inicio = time.time()
    modelo_sft = entrenar_sft(n)
    tiempos[f"SFT {n}"] = round(time.time() - inicio)
    resultados[f"SFT {n}"], detalles[f"SFT {n}"] = evaluar(modelo_sft)
    if n == 150:
        modelo_sft.save_pretrained("adapter_sft_150")
    del modelo_sft
    liberar_memoria()

pd.DataFrame(resultados).T

---

**📝 Pregunta 2 (1,0 pts):** Comparen SFT 30 con SFT 150. ¿Qué cambió en los tipos conocidos y en los tipos nuevos, y por qué? ¿Qué creen que pasaría si agregaran diez demostraciones de uno de los tipos nuevos? ¿Y si, en vez de LoRA, ajustaran todos los pesos del modelo con las mismas 150 demostraciones? Como referencia: el adapter tiene unos 2,2 millones de parámetros, el 0,45 % de los 494 millones del modelo.
> **Respuesta:**

---

# 3. GRPO con resultados

En la clase 8 vimos GRPO con recompensas verificables (RLVR): para cada pregunta el modelo genera un grupo de respuestas, una regla les pone nota y GRPO refuerza las que superan el promedio de su grupo.

Aquí la regla es `revisar`, que solo necesita el resultado correcto, no la consulta. Por eso podemos entrenar con las 600 preguntas que el SFT no puede usar, entre ellas las 172 de tipos nuevos. Entrenaremos 150 pasos: en cada paso el modelo ve 4 preguntas y genera 8 consultas para cada una, así que ve cada pregunta una sola vez.

## 3.1 La recompensa

Completen `recompensa`, que convierte la revisión de cada consulta en un puntaje: 0 si `revisar` dice `"falla"`, 0,2 si dice `"distinto"` y 1 si dice `"correcto"`.

`GRPOTrainer` la llamará con las respuestas generadas (`completions`) y con las columnas `resultado` y `ordenado` del dataset de la sección 3.3. Cada respuesta es una lista con un mensaje, y su texto está en `completion[0]["content"]`. En ese dataset, `resultado` viene como texto JSON: léanlo con `json.loads`. **(0,6 pts)**

In [ ]:
def recompensa(completions: list[list[dict]], resultado: list[str],
               ordenado: list[bool], **kwargs) -> list[float]:
    # TODO 3: recompensa por ejecución

    return [0.0] * len(completions)  # Reemplazar

In [ ]:
def como_mensaje(texto: str) -> list[dict]:
    # el formato en que GRPOTrainer entrega cada respuesta a `recompensa`;
    # sirve para llamarla a mano, aquí y en la sección 3.2
    return [{"role": "assistant", "content": texto}]


anuladas = json.dumps(n_anuladas)
regiones = json.dumps([["AN"], ["BI"], ["RM"], ["VS"]])
casos = [  # consulta, resultado correcto, ordenado, puntaje esperado
    ("SELECT COUNT(*) FROM ventas WHERE estado = 'A'", anuladas, False, 1.0),
    ("```sql\nSELECT COUNT(*) FROM ventas WHERE estado = 'A';\n```",
     anuladas, False, 1.0),
    ("SELECT COUNT(*) FROM ventas", anuladas, False, 0.2),
    ("SELECT COUNT(*) FROM ventas WHERE", anuladas, False, 0.0),
    ("SELECT DISTINCT region FROM ventas ORDER BY region DESC", regiones,
     True, 0.2),
    ("SELECT DISTINCT region FROM ventas ORDER BY region DESC", regiones,
     False, 1.0),
]
puntajes = recompensa([como_mensaje(c[0]) for c in casos],
                      [c[1] for c in casos], [c[2] for c in casos])
assert len(puntajes) == len(casos), "debe haber un puntaje por consulta"
for caso, puntaje in zip(casos, puntajes):
    assert puntaje == caso[3], (caso, puntaje)
print("Las pruebas pasaron.")

## 3.2 Un grupo de GRPO

Antes de entrenar, veamos un grupo: el modelo de SFT 150 genera ocho consultas para una pregunta con temperatura 1, como lo hará GRPO, y la función `recompensa` puntúa cada una. La celda elige la primera de 50 preguntas en que las recompensas no son todas iguales.

In [ ]:
modelo = PeftModel.from_pretrained(cargar_4bits(), "adapter_sft_150")
modelo.eval()
torch.manual_seed(SEMILLA)

for _, fila in solo_resultado.head(50).iterrows():
    texto = tokenizador.apply_chat_template(
        mensajes(fila["pregunta"]), tokenize=False,
        add_generation_prompt=True)
    entrada = tokenizador(texto, return_tensors="pt").to(0)
    with torch.no_grad():
        salida = modelo.generate(
            **entrada, max_new_tokens=64, do_sample=True, temperature=1.0,
            top_p=1.0, top_k=0, repetition_penalty=1.0,
            num_return_sequences=8, pad_token_id=tokenizador.pad_token_id)
    grupo = tokenizador.batch_decode(
        salida[:, entrada.input_ids.shape[1]:], skip_special_tokens=True)
    puntajes = recompensa([como_mensaje(g) for g in grupo],
                          [json.dumps(fila["resultado"])] * 8,
                          [fila["ordenado"]] * 8)
    if len(set(puntajes)) > 1:
        break
else:
    raise RuntimeError("En las 50 preguntas, las ocho recompensas fueron "
                       "iguales: revisen su recompensa.")

print(fila["pregunta"], "\n")
for i, (consulta, puntaje) in enumerate(zip(grupo, puntajes)):
    print(f"[{i}] recompensa {puntaje:.1f}: {consulta.strip()}")

del modelo
liberar_memoria()

---

**📝 Pregunta 3 (1,2 pts):** Respondan las tres partes.

**(a)** Con el grupo de ocho consultas de la celda anterior, calculen a mano la ventaja de cada una: (recompensa − promedio del grupo) / desviación estándar del grupo. Usen la desviación estándar de la muestra (dividiendo por n − 1), como hace `trl`. ¿Qué consultas refuerza GRPO y cuáles frena? El grupo puede cambiar de una GPU a otra: trabajen con el que les salió.

**(b)** ¿Qué pasaría con las ventajas si las ocho consultas acertaran? ¿Y qué cambiaría en el entrenamiento si la recompensa diera 0 también a las consultas que se ejecutan pero no aciertan?

**(c)** ¿Por qué partimos de SFT 150 y no del modelo base?

> **Respuesta:**

---

## 3.3 Entrenamiento

GRPO sigue entrenando el adapter de SFT 150 con las 600 preguntas que solo tienen resultado.

In [ ]:
inicio = time.time()

datos_grpo = Dataset.from_list([
    {"prompt": mensajes(fila["pregunta"]),
     "resultado": json.dumps(fila["resultado"]),
     "ordenado": fila["ordenado"]}
    for _, fila in solo_resultado.iterrows()])

# sin is_trainable=True, el adapter se carga congelado
modelo = PeftModel.from_pretrained(
    prepare_model_for_kbit_training(cargar_4bits()), "adapter_sft_150",
    is_trainable=True)

entrenador = GRPOTrainer(
    model=modelo,
    reward_funcs=recompensa,
    train_dataset=datos_grpo,
    processing_class=tokenizador,
    args=GRPOConfig(
        output_dir="grpo",
        max_steps=150,
        per_device_train_batch_size=32,  # consultas por paso
        num_generations=8,               # consultas por pregunta
        max_completion_length=64,
        temperature=1.0,
        learning_rate=5e-5,
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=SEMILLA,
    ),
)
# con fp16=True, los parámetros que se entrenan deben estar en fp32
for parametro in entrenador.model.parameters():
    if parametro.requires_grad:
        parametro.data = parametro.data.float()
entrenador.train()

tiempos["SFT 150 + GRPO"] = (tiempos["SFT 150"]
                             + round(time.time() - inicio))

La curva muestra, cada 10 pasos, la recompensa promedio y la fracción de grupos en que las ocho recompensas son iguales.

In [ ]:
curva = pd.DataFrame(entrenador.state.log_history).dropna(subset=["reward"])
plt.figure(figsize=(6, 3.5))
plt.plot(curva["step"], curva["reward"], label="recompensa promedio")
plt.plot(curva["step"], curva["frac_reward_zero_std"],
         label="grupos con recompensas iguales")
plt.xlabel("paso")
plt.legend()
plt.show()

curva[["step", "reward", "frac_reward_zero_std"]].round(2)

In [ ]:
resultados["SFT 150 + GRPO"], detalles["SFT 150 + GRPO"] = evaluar(
    entrenador.model)
del entrenador, modelo
liberar_memoria()

# 4. Comparación

In [ ]:
tabla = pd.DataFrame(resultados).T
tabla["entrenamiento (s)"] = pd.Series(tiempos).astype("Int64")
tabla

---

**📝 Pregunta 4 (1,2 pts):** Con la tabla: ¿qué agregó GRPO y a qué costo? Si la empresa ya tiene 150 consultas escritas, ¿por qué no basta con escribir más? ¿Qué pasaría si la recompensa solo revisara que la consulta se ejecuta? Si la tarea fuera redactar un correo amable, donde no hay un resultado con que comparar, ¿qué usarían en lugar de esta recompensa?
> **Respuesta:**

---